# Levenshtein Distance

In [49]:
import numpy as np

def get_sdi(ref, pred):
    r = ref.split()
    h = pred.split()
    n = len(r)

    # Construct matrix
    mtx = [[0] * (len(h)+1) for _ in range(len(r)+1)]

    for i in range(len(r)+1):
        mtx[i][0] = i  # deletions
    for j in range(len(h)+1):
        mtx[0][j] = j  # insertions

    # Fill mtx
    for i in range(1, len(r)+1):
        for j in range(1, len(h)+1):
            if r[i-1] == h[j-1]:
                mtx[i][j] = mtx[i-1][j-1]
            else:
                mtx[i][j] = 1 + min(
                    mtx[i-1][j-1],  # sub
                    mtx[i-1][j],    # del
                    mtx[i][j-1]     # ins
                )

    # Determine error type + weight based on word
    i, j = len(r), len(h)
    S, D, I = [], [], [] 
    while i > 0 or j > 0:
        if i > 0 and j > 0 and r[i-1] == h[j-1]: # No error
            i, j = i-1, j-1
            S.insert(0, 0)
            D.insert(0, 0)
            I.insert(0, 0)

        elif i > 0 and j > 0 and mtx[i][j] == mtx[i-1][j-1] + 1: # Sub
            i, j = i-1, j-1
            S.insert(0, 1)
            D.insert(0, 0)
            I.insert(0, 0)

        elif i > 0 and mtx[i][j] == mtx[i-1][j] + 1: # Del
            i -= 1
            S.insert(0, 0)
            D.insert(0, 1)
            I.insert(0, 0)

        elif j > 0 and mtx[i][j] == mtx[i][j-1] + 1: # Ins
            j -= 1
            S.insert(0, 0)
            D.insert(0, 0)
            I.insert(0, 1)
        
        errors = np.vstack((S, D, I))

    return errors

# 1. Word by Word Weighting with LLM

In [48]:
from google import genai
from dotenv import load_dotenv
import re
import os

load_dotenv()
client = genai.Client(api_key=os.getenv("GENAI_API_KEY"))

def get_weights(ref, pred, errors):
    ref_words = ref.split()
    pred_words = pred.split()

    for i in range(errors.shape[1]):
        if errors[0][i] == 1:  # Substitution
            pass
        elif errors[1][i] == 1:  # Deletion
            pred_words.insert(i, "_")
        elif errors[2][i] == 1:  # Insertion
            ref_words.insert(i, "_")
    
    ref_aligned = " ".join(ref_words)
    pred_aligned = " ".join(pred_words)

    prompt = f"""
        
        Given the following reference and prediction transcript, construct a vector of weights of the format [w_1, w_2, ..., w_N] where N is the number of words. 
        Return the weights as a comma-separated string of floats with absolutely NO additional text.
        For each word position i:
            - If ref[i] = pred[i], w_i = 0
            - If ref[i] != pred[i] and neither is "_" (substitution), w_i = weight [0,1] reflecting how dissimilar the words are (1 = completely different, 0 = very similar)
            - If ref[i] = "_" (insertion), w_i = weight [0,1] reflecting the how much adding pred[i] changes the meaning of the sentence (1 = very important, 0 = not important)
            - If pred[i] = "_" (deletion), w_i = weight [0,1] reflecting the how much removing ref[i] changes the meaning of the sentence (1 = very important, 0 = not important)

        Reference: {ref_aligned}
        Prediction: {pred_aligned}
    """

    response = client.models.generate_content(
        model="gemini-2.5-flash", 
        contents=[prompt],
    )

    # Sanity check for model output
    clean_response = re.sub(r"[^0-9,\. ]", "", response.text)

    weights = [float(w) for w in clean_response.split(",")]

    return weights

In [55]:
def weighted_wer_llm(ref, pred):
    ref_length = len(ref.split())

    # Get subs, dels, and ins errors as matrix: 
    # [[S],
    #  [D],
    #  [I]]
    errors = get_sdi(ref, pred) 

    print("Error Matrix:\n", errors)
    print("\n")

    # Get impact weights for each word 
    weights = get_weights(ref, pred, errors)

    print("Weights:\n", weights)
    print("\n")

    # Computed weighted WER
    weighted_errors = errors * weights
    sum = np.sum(weighted_errors)
    weighted_wer = sum / ref_length

    print("LLM Weighted WER:", weighted_wer)
    print("\n")

# 2. Sentencewise Weighting with Text Embedding

In [96]:
from google import genai
from dotenv import load_dotenv
import re
import os

load_dotenv()
client = genai.Client(api_key=os.getenv("GENAI_API_KEY"))

# Defines maximum error impact from similarity 
alpha = 4.0

# Sharpness of exponential asymptotes
beta = 4.0

def get_sim_factor(ref, pred):
    ref_embedding = client.models.embed_content(
        model="gemini-embedding-001",
        contents=ref
    ).embeddings[0].values
    
    pred_embedding = client.models.embed_content(
        model="gemini-embedding-001",
        contents=pred
    ).embeddings[0].values

    lin_sim = np.dot(ref_embedding, pred_embedding) / (np.linalg.norm(ref_embedding) * np.linalg.norm(pred_embedding))
    norm_sim = 2 * (lin_sim - 0.5) if lin_sim > 0.5 else 0 # Most similarity values lie between 0.5 and 1.0


    exp_sim = alpha * np.exp(- beta * norm_sim)

    return exp_sim
    

In [97]:
def weighted_wer_emb(ref, pred):
    ref_length = len(ref.split())

    # Get errors
    errors = get_sdi(ref, pred)

    # Calculate normal WER
    sum = np.sum(errors)
    wer = sum / ref_length
    print("Standard WER: ", wer)

    # Weight by whole sentence similarity (pred vs. ref) --> higher similarity = lower error
    sim_weighting = get_sim_factor(ref, pred)
    print("Similarity score: ", sim_weighting)

    weighted_wer = sim_weighting * wer

    print("EMB Weighted WER:", np.round(weighted_wer, 2))
    print("\n")


# Usage

In [ ]:
# Using word-by-word LLM weighting

ref = "I want to sleep"
pred1 = "I want to sheep"
pred2 = "I want sleep"

print("------------ WITH PRED 1 --------------")
weighted_wer_llm(ref, pred1)

print("------------ WITH PRED 2 --------------")
weighted_wer_llm(ref, pred2)

In [101]:
# Using sentence-wise embedding weighting

ref = "I want to sleep"
pred1 = "I want to sheep"
pred2 = "I want sleep"

print("------------ WITH PRED 1 --------------")
weighted_wer_emb(ref, pred1)

print("------------ WITH PRED 2 --------------")
weighted_wer_emb(ref, pred2)

------------ WITH PRED 1 --------------
Standard WER:  0.25
Similarity score:  0.6687878760084761
EMB Weighted WER: 0.17


------------ WITH PRED 2 --------------
Standard WER:  0.25
Similarity score:  0.13385607317148843
EMB Weighted WER: 0.03


